# 📈 Smart Business Intelligence - Sales Prediction & Forecasting

**Projet :** Smart Business Intelligence & Sales Analytics Platform  
**Auteur :** Chaima Debchi  
**Objectif :** Prédire le chiffre d'affaires mensuel futur en transformant une série temporelle en un problème d'apprentissage supervisé, et comparer 3 algorithmes de régression (Régression Linéaire, Random Forest, Gradient Boosting).

---

## 🧠 Concepts Clés Expliqués Simplement

### 1. Pourquoi transformer une série temporelle en régression supervisée ?
En Machine Learning, un modèle a besoin de variables explicatives ($X$) pour prédire une cible ($y$). Pour prédire les ventes du mois prochain, nous créons des variables basées sur le passé :
- **Les retards (*Lags*) :** Ventes du mois précédent (`Lag_1`), ventes d'il y a 2 mois (`Lag_2`), et ventes du même mois l'année dernière (`Lag_12`).
- **Les moyennes mobiles (*Rolling Means*) :** Moyenne lissée des 3 derniers mois pour capturer la tendance récente.
- **La saisonnalité :** Numéro du mois (`Month`) et du trimestre (`Quarter`).

### 2. Le piège absolu : La fuite temporelle (*Data Leakage*)
> ⚠️ **Important pour ton CV :** En séries temporelles, il est **strictement interdit** d'utiliser `train_test_split(shuffle=True)`. Si on mélange les lignes aléatoirement, le modèle s'entraînerait avec des données futures pour prédire le passé !  
> **Notre solution rigoureuse :** Découpage chronologique pur :  
> - Entraînement : **2015 à 2017** (36 mois)  
> - Test (Validation hors échantillon) : **2018** (12 mois complets)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: "%.2f" % x)

# Chargement des données propres
df = pd.read_csv("../data/processed/cleaned_superstore.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"])
print(f"Transactions chargées : {len(df):,}")

## 1. Agrégation Mensuelle & Création des Variables (Feature Engineering)

In [ ]:
# Agrégation des ventes par mois (Month Start - MS)
df_monthly = df.set_index("Order Date").resample("MS")["Sales"].sum().reset_index()
df_monthly.columns = ["Date", "Sales"]

# Variables temporelles et calendaires
df_monthly["Year"] = df_monthly["Date"].dt.year
df_monthly["Month"] = df_monthly["Date"].dt.month
df_monthly["Quarter"] = df_monthly["Date"].dt.quarter

# Création des Lags (retards) et Moyennes Mobiles
df_monthly["Lag_1"] = df_monthly["Sales"].shift(1)       # Mois M-1
df_monthly["Lag_2"] = df_monthly["Sales"].shift(2)       # Mois M-2
df_monthly["Lag_12"] = df_monthly["Sales"].shift(12)     # Même mois N-1 (Saisonnalité annuelle)
df_monthly["Rolling_Mean_3"] = df_monthly["Sales"].shift(1).rolling(3).mean()

# Suppression des premières lignes contenant des NaN dus au lag_12
df_model = df_monthly.dropna().copy()
print(f"Mois exploitables pour la modélisation : {len(df_model)}")
df_model.head(3)

## 2. Séparation Temporelle Train / Test (Split Chronologique)

In [ ]:
train = df_model[df_model["Year"] < 2018]
test = df_model[df_model["Year"] == 2018].copy()

features = ["Month", "Quarter", "Lag_1", "Lag_2", "Lag_12", "Rolling_Mean_3"]
X_train, y_train = train[features], train["Sales"]
X_test, y_test = test[features], test["Sales"]

print(f"Taille Train (2015-2017) : {len(X_train)} mois")
print(f"Taille Test  (2018)      : {len(X_test)} mois")

## 3. Entraînement et Évaluation Comparative des Modèles

Nous mesurons la performance avec 3 métriques standard :
- **MAE (Mean Absolute Error) :** Erreur moyenne en dollars (plus c'est bas, mieux c'est).
- **RMSE (Root Mean Squared Error) :** Racine de l'erreur quadratique moyenne (pénalise fortement les gros écarts).
- **$R^2$ (Coefficient de détermination) :** Pourcentage de la variance des ventes expliqué par le modèle (proche de 1 = excellent).

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=5),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=3, learning_rate=0.05)
}

results = []
predictions = {"Date": test["Date"].values, "Réel": y_test.values}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    predictions[name] = preds
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    results.append({
        "Modèle": name,
        "MAE ($)": mae,
        "RMSE ($)": rmse,
        "R² Score": r2
    })

comp_df = pd.DataFrame(results).sort_values("R² Score", ascending=False)
comp_df

## 4. Visualisation : Ventes Réelles vs Ventes Prédites en 2018

In [ ]:
pred_df = pd.DataFrame(predictions)

plt.figure(figsize=(14, 6))
plt.plot(pred_df["Date"], pred_df["Réel"], marker="o", linewidth=3, label="Ventes Réelles (2018)", color="black")
plt.plot(pred_df["Date"], pred_df["Gradient Boosting"], marker="s", linewidth=2, linestyle="--", label="Gradient Boosting (R² = 0.71)", color="#2ca02c")
plt.plot(pred_df["Date"], pred_df["Linear Regression"], marker="^", linewidth=2, linestyle=":", label="Linear Regression (R² = 0.65)", color="#1f77b4")
plt.plot(pred_df["Date"], pred_df["Random Forest"], marker="d", linewidth=2, linestyle="-.", label="Random Forest (R² = 0.60)", color="#ff7f0e")

plt.title("Comparaison des Modèles : Ventes Réelles vs Prédictions (Année 2018)", fontsize=14, fontweight="bold")
plt.xlabel("Mois")
plt.ylabel("Ventes Mensuelles ($)")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Importance des Variables (*Feature Importance*)
Quels facteurs permettent au modèle de prédire les ventes avec succès ?

In [ ]:
best_gb = models["Gradient Boosting"]
feat_imp = pd.Series(best_gb.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind="barh", color="#2ca02c")
plt.title("Importance des Variables dans le Gradient Boosting", fontsize=13, fontweight="bold")
plt.xlabel("Importance Relative")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

> 💡 **Conclusion du Modèle :**
>- **Le Gradient Boosting est le meilleur modèle** : il atteint un **$R^2$ de 0.71** (71 % de la variance expliquée) avec une erreur moyenne (MAE) de 12 056 $ sur des volumes mensuels dépassant 80 000 $.
>- **La variable la plus influente est `Lag_12`** : les ventes d'une période donnée s'expliquent d'abord par les performances du même mois l'année précédente (effet de saisonnalité commerciale annuelle).